In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import torch

print("Environment Ready 🚀")
print("GPU Available:", torch.cuda.is_available())


Environment Ready 🚀
GPU Available: False


In [2]:
# Step 1: Generate random binary challenges

def generate_challenges(num_samples, num_stages):
    return np.random.randint(0, 2, size=(num_samples, num_stages))

# Example: 5 challenges, 8-stage PUF
challenges = generate_challenges(5, 8)
print(challenges)


[[1 0 0 0 0 0 1 0]
 [1 0 1 1 0 0 1 0]
 [1 0 0 0 0 1 0 0]
 [1 1 1 0 1 0 1 0]
 [1 1 0 1 1 0 0 0]]


In [3]:
# Step 2: Convert challenge to feature vector Phi

def challenge_to_phi(challenges):
    num_samples, num_stages = challenges.shape
    phi = np.ones((num_samples, num_stages))

    for i in range(num_samples):
        for j in range(num_stages):
            product = 1
            for k in range(j, num_stages):
                product *= (1 - 2 * challenges[i][k])
            phi[i][j] = product

    return phi

# Test it
phi = challenge_to_phi(challenges)
print(phi)


[[ 1. -1. -1. -1. -1. -1. -1.  1.]
 [ 1. -1. -1.  1. -1. -1. -1.  1.]
 [ 1. -1. -1. -1. -1. -1.  1.  1.]
 [-1.  1. -1.  1.  1. -1. -1.  1.]
 [ 1. -1.  1.  1. -1.  1.  1.  1.]]


In [4]:
# Step 3: Create weight vector (device-specific delays)

def generate_weights(num_stages):
    return np.random.randn(num_stages)

weights = generate_weights(8)  # 8-stage example
print(weights)


[-0.30832376 -0.46369281 -0.86486378  0.04744811  0.36792299  0.0965671
 -0.28055593 -1.00540064]


In [5]:
# Step 4: Generate PUF response

def generate_response(phi, weights):
    # Compute dot product
    delays = np.dot(phi, weights)
    
    # Convert to binary response
    responses = (delays > 0).astype(int)
    
    return responses

responses = generate_response(phi, weights)
print("Responses:")
print(responses)


Responses:
[0 0 0 1 0]


In [6]:
    # Step 5: Generate large CRP dataset

num_samples = 50000   # 50k CRPs
num_stages = 32       # 32-stage Arbiter PUF

# Generate challenges
X_challenges = generate_challenges(num_samples, num_stages)

# Convert to Phi
X_phi = challenge_to_phi(X_challenges)

# Generate new device weights
weights = generate_weights(num_stages)

# Generate responses
y_responses = generate_response(X_phi, weights)

print("Dataset shape:")
print("X:", X_phi.shape)
print("y:", y_responses.shape)


Dataset shape:
X: (50000, 32)
y: (50000,)


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_phi, y_responses, test_size=0.3, random_state=42
)

# Train Logistic Regression (linear model)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print("Attack Accuracy:", accuracy)


Attack Accuracy: 0.9983333333333333


In [8]:
# Step 1: XOR Arbiter PUF

def generate_xor_response(phi, num_xor):
    num_samples, num_stages = phi.shape
    
    # Generate multiple weight vectors (one per Arbiter)
    weight_list = [generate_weights(num_stages) for _ in range(num_xor)]
    
    # Compute responses for each arbiter
    all_responses = []
    
    for weights in weight_list:
        delays = np.dot(phi, weights)
        responses = (delays > 0).astype(int)
        all_responses.append(responses)
    
    # XOR all responses together
    xor_response = all_responses[0]
    for i in range(1, num_xor):
        xor_response = np.logical_xor(xor_response, all_responses[i]).astype(int)
    
    return xor_response


In [11]:
# Step 2: Generate XOR dataset

num_samples = 50000
num_stages = 32
num_xor = 3

# Generate challenges
X_challenges = generate_challenges(num_samples, num_stages)
X_phi = challenge_to_phi(X_challenges)

# Generate XOR responses
y_xor = generate_xor_response(X_phi, num_xor)

print("X shape:", X_phi.shape)
print("y shape:", y_xor.shape)


X shape: (50000, 32)
y shape: (50000,)


In [12]:
# Step 3: Attack XOR PUF

X_train, X_test, y_train, y_test = train_test_split(
    X_phi, y_xor, test_size=0.3, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("XOR PUF Attack Accuracy:", accuracy)


XOR PUF Attack Accuracy: 0.5244


In [13]:
#Multi Layer Perceptron
from sklearn.neural_network import MLPClassifier

# Train MLP (non-linear model)
mlp = MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=20)

mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

print("MLP Attack Accuracy:", accuracy_mlp)


MLP Attack Accuracy: 0.9712666666666666


C:\Users\divis\PUF_ML_PROJECT\venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
